# Praktikum Pertemuan 3 — Menyiapkan Data Proyek agar Layak dan Aman Dianalisis

**Mata Kuliah:** Exploratory Data Analysis (KMTI21133) — Pertemuan 3  
**Fase PjBL:** Perencanaan  
**Sub-CPMK3:** Mahasiswa mampu merumuskan pertanyaan analisis dari masalah nyata serta memperoleh data
sesuai lisensi sumbernya, serta mengidentifikasi dan menangani data pribadi yang terkandung di dalamnya (C4, A3).

**Cara kerja.** Sel bertanda `# TODO` Anda kerjakan sendiri; setiap TODO disertai *expected output* agar Anda bisa memeriksa jawaban sendiri. Sel Markdown bertanda **Jawaban Anda** diisi dengan tulisan, bukan kode.

**Bagian A–F** dikerjakan di kelas dengan dataset latihan. **Bagian G** dikerjakan bersama tim di luar kelas dengan dataset proyek Anda sendiri, dan hasilnya menjadi isi **M1 (bobot 5%)** yang dikumpulkan sesuai tenggat di LMS.

## Bagian 0 — Persiapan

Dataset latihan: **`peserta_pelatihan_umkm_2025.csv`** — 620 baris, 28 kolom.
Data pendaftaran dan hasil pelatihan UMKM binaan kampus.

> **Catatan penting:** seluruh isi dataset ini **sintetis** dan dibuat untuk latihan. Tidak ada orang nyata di dalamnya.
> Nomor NIK, nomor telepon, dan surel sengaja dibuat dengan pola yang tidak valid. Perlakukan tetap seolah-olah data nyata,
> karena itulah kebiasaan yang sedang kita latih.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
print('pandas', pd.__version__)

In [ ]:
import os

FILE = 'peserta_pelatihan_umkm_2025.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()          # pilih peserta_pelatihan_umkm_2025.csv
    except ImportError:
        raise FileNotFoundError('/content/drive/MyDrive/Semester 3/EDA2026/Pertemuan03/Data/peserta_pelatihan_umkm_2025.csv')

df = pd.read_csv(FILE)
print(df.shape)
df.head(3)

TypeError: 'NoneType' object is not subscriptable

---
## Bagian A — Merumuskan pertanyaan analisis

Sebuah pertanyaan analisis dianggap layak bila lolos **tiga uji**:

1. **Uji kolom** — kolom mana yang akan menjawabnya?
2. **Uji bentuk jawaban** — jawabannya berupa angka, peringkat, perbandingan, atau sebaran?
3. **Uji tindak lanjut** — kalau terjawab, apa yang bisa diputuskan?

In [1]:
# TODO A1 — kenali kolom yang tersedia sebelum menulis pertanyaan
# a. Cetak daftar nama kolom beserta tipe datanya
print(df.dtypes)

# b. Cetak jumlah baris dan kolom
print(df.shape[0], 'baris x', df.shape[1], 'kolom')

# Expected: 620 baris x 28 kolom

NameError: name 'df' is not defined

### TODO A2 — perbaiki tiga pertanyaan berikut

Tulis versi perbaikannya di sel di bawah, beserta kolom yang akan dipakai.

| No | Draf pertanyaan | Masalahnya |
|---|---|---|
| 1 | Bagaimana kondisi peserta pelatihan? | Terlalu kabur |
| 2 | Apakah peserta yang lebih kaya lebih sukses? | "Kaya" dan "sukses" belum didefinisikan sebagai kolom |
| 3 | Apakah pelatihan menaikkan omzet peserta? | Uji kolom gagal — cek sendiri mengapa |

**Jawaban Anda:**

1. Perbaikan: Berapa rata-rata kenaikan skor (`skor_pascates` dikurangi `skor_prates`) peserta pada tiap `kategori_usaha`, sehingga terlihat kategori mana yang perlu pendampingan tambahan? — Kolom: `kategori_usaha`, `skor_prates`, `skor_pascates`
2. Perbaikan: Apakah peserta dengan `omzet_usaha_bulanan` (saat mendaftar) pada kuartil tertinggi memiliki rata-rata `skor_pascates` lebih tinggi daripada kuartil terendah? ("Kaya" = omzet awal, "sukses" = skor pascates.) — Kolom: `omzet_usaha_bulanan`, `skor_pascates`, `skor_prates`
3. Mengapa uji kolom gagal: Dataset hanya memiliki omzet saat mendaftar (sebelum pelatihan) dan tidak ada omzet sesudah pelatihan. Tanpa dua titik waktu, kenaikan omzet tidak bisa dihitung. Pertanyaan ini baru bisa dijawab jika ada data omzet pascapelatihan. Dengan data yang ada, yang bisa diukur adalah kenaikan skor.

In [ ]:
# TODO A3 — buktikan jawaban nomor 3 dengan kode
# Periksa apakah ada kolom omzet SESUDAH pelatihan di dataset ini.
# Petunjuk: cari nama kolom yang memuat kata 'omzet'
# Expected: hanya ada satu kolom omzet, yaitu omzet saat mendaftar


In [ ]:
kolom_omzet = [c for c in df.columns if 'omzet' in c.lower()]
print(kolom_omzet)
print('Jumlah kolom omzet:', len(kolom_omzet))

['omzet_usaha_bulanan']
Jumlah kolom omzet: 1


---
## Bagian B — Skala pengukuran

### TODO B1 — klasifikasikan delapan kolom berikut

Isi tabel di sel Markdown ini. Pilihan: **nominal · ordinal · interval · rasio**.



| Kolom | Skala | Alasan singkat |
|---|---|---|
| `kategori_usaha` | nominal | Hanya label kelompok, tidak ada urutan |
| `pendidikan_terakhir` | ordinal | Ada urutan jenjang, tetapi jarak antarjenjang tidak sama |
| `skor_prates` | interval | Selisih skor bermakna, tetapi nol bukan berarti "tanpa pengetahuan sama sekali" |
| `omzet_usaha_bulanan` | rasio | Nol berarti tidak ada omzet, dan perbandingan "dua kali lipat" bermakna |
| `jumlah_karyawan` | rasio | Hasil hitungan dengan nol mutlak |
| `tingkat_kepuasan` | ordinal | Skala Likert: berurutan, tetapi jaraknya tidak pasti sama |
| `tanggal_daftar` | interval | Selisih tanggal bermakna, tetapi tidak ada nol alami |
| `kecamatan` | nominal | Label wilayah, tidak ada urutan |

In [ ]:
# TODO B2 — perbaiki satu kolom ordinal yang salah tipe
# a. Jadikan 'pendidikan_terakhir' bertipe kategori berurutan
#    Urutan: SD < SMP < SMA/SMK < D3 < S1 < S2
# b. Tampilkan value_counts() yang sudah urut menurut jenjang (bukan abjad)
# c. Cetak median jenjang pendidikan lewat .cat.codes
# Expected: median = SMA/SMK


In [ ]:
urut = ['SD', 'SMP', 'SMA/SMK', 'D3', 'S1', 'S2']
df['pendidikan_terakhir'] = pd.Categorical(df['pendidikan_terakhir'],
                                           categories=urut, ordered=True)

# b. urut menurut jenjang, bukan abjad
print(df['pendidikan_terakhir'].value_counts().sort_index())

# c. median lewat kode kategori (kode -1 = nilai tak dikenal/kosong, dibuang)
kode = df['pendidikan_terakhir'].cat.codes
kode = kode[kode >= 0]
print('Median jenjang:', urut[int(kode.median())])

pendidikan_terakhir
SD          24
SMP         71
SMA/SMK    281
D3         107
S1         124
S2          13
Name: count, dtype: int64
Median jenjang: SMA/SMK


In [ ]:
# TODO B3 — ubah dua kolom tanggal menjadi bertipe datetime
# Kolom: tanggal_lahir dan tanggal_daftar
# Lalu cetak tanggal pendaftaran paling awal dan paling akhir
# Expected: rentang pendaftaran berada di tahun 2025


In [ ]:
for k in ['tanggal_lahir', 'tanggal_daftar']:
    df[k] = pd.to_datetime(df[k], errors='coerce')

print('Daftar paling awal :', df['tanggal_daftar'].min())
print('Daftar paling akhir:', df['tanggal_daftar'].max())

Daftar paling awal : 2025-02-01 00:00:00
Daftar paling akhir: 2025-05-31 00:00:00


---
## Bagian C — Kamus data

Kamus data adalah satu baris penjelasan untuk setiap kolom. Bangkitkan kerangkanya dengan kode, isi maknanya sendiri.

In [ ]:
# TODO C1 — bangkitkan kerangka kamus data
# Buat DataFrame bernama 'kamus' dengan kolom:
#   nama_kolom, tipe_data, nilai_unik, nilai_kosong, contoh_nilai
# lalu tambahkan empat kolom kosong yang akan diisi manual:
#   deskripsi, skala_pengukuran, kategori_data_pribadi, tindakan_penanganan
# Expected: 28 baris x 9 kolom


In [ ]:
kamus = pd.DataFrame({
    'nama_kolom'  : df.columns,
    'tipe_data'   : df.dtypes.astype(str).values,
    'nilai_unik'  : df.nunique().values,
    'nilai_kosong': df.isna().sum().values,
    'contoh_nilai': [df[c].dropna().iloc[0] if df[c].notna().any() else None
                     for c in df.columns],
})
for k in ['deskripsi', 'skala_pengukuran', 'kategori_data_pribadi', 'tindakan_penanganan']:
    kamus[k] = ''

print(kamus.shape)   # 28 x 9
kamus.head()

(28, 9)


,nama_kolom,tipe_data,nilai_unik,nilai_kosong,contoh_nilai,deskripsi,skala_pengukuran,kategori_data_pribadi,tindakan_penanganan
0,id_peserta,object,620,0,PLT-2025-1309,,,,
1,nama_lengkap,object,450,0,Ratna Hidayat,,,,
2,nik_dummy,int64,620,0,99991405899249,,,,
3,no_hp,float64,611,9,80004260813.0,,,,
4,email,object,450,0,ratna.hidayat@contoh.example,,,,


In [ ]:
# TODO C2 — kolom mana yang punya nilai kosong?
# Tampilkan hanya baris kamus yang nilai_kosong-nya lebih dari 0
# Expected: tepat 3 kolom yang punya nilai kosong


In [ ]:
kamus[kamus['nilai_kosong'] > 0]

,nama_kolom,tipe_data,nilai_unik,nilai_kosong,contoh_nilai,deskripsi,skala_pengukuran,kategori_data_pribadi,tindakan_penanganan
3,no_hp,float64,611,9,80004260813.0,,,,
26,skor_pascates,float64,64,34,67.0,,,,
27,tingkat_kepuasan,float64,28,18,5.0,,,,


### TODO C3 — isi kamus untuk lima kolom

Lengkapi tabel berikut. Deskripsi ditulis sebagai kalimat lengkap, bukan pengulangan nama kolom.
| Kolom | Deskripsi | Skala | Satuan |
|---|---|---|---|
| `lama_usaha_tahun` | Lama usaha peserta telah berjalan sejak berdiri hingga saat mendaftar pelatihan. | rasio | tahun |
| `kehadiran_persen` | Persentase pertemuan pelatihan yang dihadiri peserta dari seluruh pertemuan yang dijadwalkan. | rasio | persen (%) |
| `skor_pascates` | Nilai tes yang dikerjakan peserta setelah pelatihan selesai untuk mengukur penguasaan materi. | interval | poin |
| `kanal_pendaftaran` | Saluran yang dipakai peserta untuk mendaftar pelatihan (misalnya daring atau datang langsung). | nominal | tanpa satuan |
| `jumlah_anak` | Banyaknya anak yang dimiliki peserta, dan tergolong data spesifik karena menyangkut data anak. | rasio | orang |

---
## Bagian D — Menandai data pribadi

| Kategori | Artinya |
|---|---|
| **pengenal langsung** | Sendirian sudah menunjuk ke satu orang |
| **quasi-identifier** | Sendirian aman, gabungannya menunjuk ke satu orang |
| **data spesifik** | Bocornya merugikan lebih berat (kesehatan, keuangan pribadi, data anak) |
| **bukan data pribadi** | Tidak menunjuk ke orang; inilah bahan analisis |

In [ ]:
# TODO D1 — susun tabel penandaan Anda sendiri
# Buat dictionary bernama PENANDAAN berisi pasangan:
#     'nama_kolom': ('kategori', 'tindakan')
# Tindakan yang tersedia: 'hapus', 'hash', 'generalisasi', 'pertahankan'
# Kolom yang tidak Anda daftarkan dianggap 'bukan data pribadi' + 'pertahankan'.
#
# Lalu bangun DataFrame 'tanda' dengan kolom: nama_kolom, kategori, tindakan
# dan cetak jumlah kolom per kategori.
# Expected: pengenal langsung >= 5 kolom; data spesifik >= 3 kolom


In [ ]:
LANGSUNG = {'nik', 'telepon', 'telp', 'hp', 'wa', 'whatsapp', 'email', 'surel',
            'alamat', 'rekening', 'ktp', 'foto'}
SPESIFIK = {'anak', 'disabilitas', 'kesehatan', 'penyakit', 'penghasilan',
            'pendapatan', 'pinjaman', 'utang', 'gaji', 'agama'}

PENANDAAN = {}
for c in df.columns:
    t = set(c.lower().split('_'))
    if c.lower() in ('nama', 'nama_lengkap', 'nama_peserta') or (t & LANGSUNG):
        PENANDAAN[c] = ('pengenal langsung', 'hapus')
    elif t & SPESIFIK:
        PENANDAAN[c] = ('data spesifik', 'pertahankan' if 'anak' in t else 'generalisasi')

# keputusan manual (menimpa hasil deteksi otomatis bila perlu)
PENANDAAN.update({
    'id_peserta'         : ('pengenal langsung', 'hash'),
    'nama_usaha'         : ('pengenal langsung', 'hapus'),
    'tanggal_lahir'      : ('quasi-identifier', 'generalisasi'),
    'kelurahan'          : ('quasi-identifier', 'generalisasi'),
    'kecamatan'          : ('quasi-identifier', 'generalisasi'),
    'jenis_kelamin'      : ('quasi-identifier', 'pertahankan'),
    'pendidikan_terakhir': ('quasi-identifier', 'pertahankan'),
})
PENANDAAN = {k: v for k, v in PENANDAAN.items() if k in df.columns}

tanda = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in PENANDAAN.items()],
    columns=['nama_kolom', 'kategori', 'tindakan'])

print(tanda['kategori'].value_counts())
print()
print(tanda.to_string(index=False))
print('\nKolom yang belum terdaftar (dianggap bukan data pribadi):')
print([c for c in df.columns if c not in PENANDAAN])

kategori
pengenal langsung    7
quasi-identifier     5
data spesifik        3
Name: count, dtype: int64

                 nama_kolom          kategori     tindakan
               nama_lengkap pengenal langsung        hapus
                  nik_dummy pengenal langsung        hapus
                      no_hp pengenal langsung        hapus
                      email pengenal langsung        hapus
               alamat_jalan pengenal langsung        hapus
         status_disabilitas     data spesifik generalisasi
                jumlah_anak     data spesifik  pertahankan
penghasilan_pribadi_bulanan     data spesifik generalisasi
                 id_peserta pengenal langsung         hash
                 nama_usaha pengenal langsung        hapus
              tanggal_lahir  quasi-identifier generalisasi
                  kelurahan  quasi-identifier generalisasi
                  kecamatan  quasi-identifier generalisasi
              jenis_kelamin  quasi-identifier  pertahankan
        pe

### TODO D2 — pertanggungjawabkan tiga keputusan

Tiga kolom berikut sering diperdebatkan. Tulis keputusan Anda **beserta alasannya**.

| Kolom | Kategori pilihan Anda | Tindakan | Alasan |
|---|---|---|---|
| `nama_usaha` | | | |
| `omzet_usaha_bulanan` | | | |
| `id_peserta` | | | |

> Petunjuk untuk `nama_usaha`: bayangkan seseorang mencari nama itu di mesin pencari.
> Petunjuk untuk `omzet_usaha_bulanan`: apakah ini data tentang **orang** atau tentang **usaha**?
> Petunjuk untuk `id_peserta`: apakah Anda masih perlu membedakan satu baris dari baris lain?

| Kolom | Kategori pilihan Anda | Tindakan | Alasan |
|---|---|---|---|
| `nama_usaha` | pengenal langsung | hapus | Nama usaha UMKM mudah dicari di mesin pencari, Google Maps, atau marketplace, dan langsung menunjuk ke pemiliknya beserta alamatnya. Nama ini tidak dibutuhkan untuk analisis. |
| `omzet_usaha_bulanan` | bukan data pribadi | pertahankan | Ini data tentang usaha, bukan tentang orang, dan menjadi bahan analisis utama. Aman selama `nama_usaha` dihapus, sehingga angka ini tidak bisa ditautkan ke pemilik. Nilai ekstrem tetap perlu dicek saat dilaporkan. |
| `id_peserta` | pengenal langsung (pseudonim) | hash | ID asli tidak perlu diketahui, tetapi baris tetap harus bisa dibedakan dan digabung dengan tabel lain. Hash menghasilkan kode yang konsisten tanpa membuka ID asli. |

---
## Bagian E — Menerapkan tiga tindakan

In [ ]:
# TODO E1 — HAPUS
# Buat DataFrame baru bernama 'aman' yang sudah membuang semua kolom bertindakan 'hapus'.
# Cetak berapa kolom yang dibuang dan berapa yang tersisa.



In [ ]:
kolom_hapus = [c for c, (k, t) in PENANDAAN.items() if t == 'hapus']
aman = df.drop(columns=kolom_hapus).copy()
print('Kolom dibuang :', len(kolom_hapus), kolom_hapus)
print('Kolom tersisa :', aman.shape[1])

Kolom dibuang : 6 ['nama_lengkap', 'nik_dummy', 'no_hp', 'email', 'alamat_jalan', 'nama_usaha']
Kolom tersisa : 22


In [ ]:
# TODO E2 — HASH
# Tulis fungsi samarkan(nilai, garam) yang mengembalikan 12 karakter pertama
# dari sha256(garam + nilai). Kembalikan np.nan bila nilainya kosong.
# Terapkan pada kolom yang Anda beri tindakan 'hash'.
# Expected: nilai asli dan hasil hash berbeda, dan nilai asli yang sama menghasilkan hash yang sama

GARAM = 'ganti-dengan-kalimat-rahasia-tim'


In [ ]:
GARAM = 'ganti-dengan-kalimat-rahasia-tim'

def samarkan(nilai, garam):
    if pd.isna(nilai):
        return np.nan
    return hashlib.sha256((garam + str(nilai)).encode('utf-8')).hexdigest()[:12]

kolom_hash = [c for c, (k, t) in PENANDAAN.items() if t == 'hash']
for c in kolom_hash:
    aman[c] = aman[c].apply(lambda v: samarkan(v, GARAM))

print('Asli :', df[kolom_hash].head(3).values.tolist())
print('Hash :', aman[kolom_hash].head(3).values.tolist())

Asli : [['PLT-2025-1309'], ['PLT-2025-1512'], ['PLT-2025-1360']]
Hash : [['09b72e9a47af'], ['565439020ae0'], ['2398346a6926']]


In [ ]:
# TODO E3 — buktikan hash bersifat konsisten dan tidak terbaca
# a. Hash dua kali nilai yang sama, tunjukkan hasilnya identik
# b. Hash dua nilai berbeda, tunjukkan hasilnya berbeda



In [ ]:
contoh = df[kolom_hash[0]].dropna().iloc[0]
contoh2 = df[kolom_hash[0]].dropna().iloc[1]

# a. nilai sama -> hash sama
print(samarkan(contoh, GARAM), samarkan(contoh, GARAM))
print('Identik:', samarkan(contoh, GARAM) == samarkan(contoh, GARAM))

# b. nilai berbeda -> hash berbeda
print(samarkan(contoh, GARAM), samarkan(contoh2, GARAM))
print('Berbeda:', samarkan(contoh, GARAM) != samarkan(contoh2, GARAM))

09b72e9a47af 09b72e9a47af
Identik: True
09b72e9a47af 565439020ae0
Berbeda: True


**Jawaban Anda:** kalau `GARAM` ikut ter-*commit* ke GitHub, apa yang masih terlindungi dan apa yang tidak?

Kalau `GARAM` ikut ter-*commit*, hash masih tidak bisa "dibaca balik" secara langsung, dan data mentah memang tidak ikut terunggah. Namun garam tidak lagi rahasia, sehingga siapa pun bisa menghitung ulang hash dari daftar kemungkinan nilai (misalnya semua nomor telepon atau ID yang polanya mudah ditebak) lalu mencocokkannya dengan data yang diunggah. Jadi yang masih melindungi hanya sulitnya menebak nilai asli. Untuk nilai berpola pendek seperti ID berurutan, nomor telepon, atau NIK, perlindungan itu hampir hilang. Garam juga tetap tersimpan di riwayat Git walau barisnya dihapus kemudian.

In [ ]:
# TODO E4 — GENERALISASI
# a. Hitung usia peserta pada tanggal acuan 1 September 2025
# b. Buat kolom 'kelompok_umur' dengan batas: <25, 25-34, 35-44, 45-54, 55+
# c. Buang kolom tanggal_lahir dan kelurahan dari 'aman'
# d. Rapikan kolom kecamatan yang sebagian tertulis KAPITAL
# Expected: kelompok terbanyak adalah 35-44; kecamatan punya tepat 10 nilai unik

ACUAN = pd.Timestamp('2025-09-01')


In [ ]:
ACUAN = pd.Timestamp('2025-09-01')

# pastikan tanggal_lahir bertipe datetime (aman dari urutan eksekusi sel)
aman['tanggal_lahir'] = pd.to_datetime(aman['tanggal_lahir'], errors='coerce')

# a. usia pada tanggal acuan
lahir = aman['tanggal_lahir']
sudah_ulang_tahun = (lahir.dt.month < ACUAN.month) | \
                    ((lahir.dt.month == ACUAN.month) & (lahir.dt.day <= ACUAN.day))
usia = ACUAN.year - lahir.dt.year - (~sudah_ulang_tahun).astype(int)

# b. kelompok umur
aman['kelompok_umur'] = pd.cut(usia, bins=[-1, 24, 34, 44, 54, 200],
                               labels=['<25', '25-34', '35-44', '45-54', '55+'])

# c. buang tanggal_lahir dan kelurahan (usia persisnya tidak disimpan)
aman = aman.drop(columns=['tanggal_lahir', 'kelurahan'])

# d. rapikan kecamatan
aman['kecamatan'] = aman['kecamatan'].str.strip().str.title()

print(aman['kelompok_umur'].value_counts().sort_index())
print('Kecamatan unik:', aman['kecamatan'].nunique())

kelompok_umur
<25       53
25-34    153
35-44    267
45-54    108
55+       39
Name: count, dtype: int64
Kecamatan unik: 10


---
## Bagian F — Memeriksa sisa risiko

Menghapus nama belum tentu cukup. Untuk setiap kombinasi quasi-identifier, hitung berapa orang yang punya kombinasi sama. Nilai terkecil disebut **k**. Target kelas ini: **k ≥ 5**.

In [ ]:
# TODO F1 — tulis fungsi cek_k(data, kolom)
# Kembalikan Series berisi: k (grup terkecil), grup berisi 1 orang, total grup



In [ ]:
def cek_k(data, kolom):
    g = data.groupby(kolom, observed=True, dropna=False).size()
    return pd.Series({
        'k (grup terkecil)'   : g.min(),
        'grup berisi 1 orang' : (g == 1).sum(),
        'total grup'          : len(g),
    })

In [ ]:
# TODO F2 — ukur risiko SEBELUM penanganan
# Gunakan data asli df dengan kolom:
#   ['kelurahan', 'jenis_kelamin', 'pendidikan_terakhir', 'tanggal_lahir']
# Expected: k = 1 dan SELURUH 620 baris berada di grup berisi satu orang


In [ ]:
cek_k(df, ['kelurahan', 'jenis_kelamin', 'pendidikan_terakhir', 'tanggal_lahir'])

,0
k (grup terkecil),1
grup berisi 1 orang,620
total grup,620


**Jawaban Anda:** keempat kolom itu tidak memuat nama maupun nomor telepon. Jelaskan dalam dua kalimat mengapa hasilnya tetap berbahaya.

 Meski tanpa nama dan nomor telepon, kombinasi kelurahan, jenis kelamin, pendidikan, dan tanggal lahir bersifat unik untuk setiap orang (k = 1), sehingga siapa pun yang punya data pembanding, seperti daftar warga atau media sosial, bisa menautkannya kembali ke satu individu. Kolom yang aman bila berdiri sendiri bisa menjadi pengenal bila digabung (*re-identification*).

In [ ]:
# TODO F3 — ukur risiko SESUDAH generalisasi
# Gunakan 'aman' dengan kolom:
#   ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'kelompok_umur']
# Expected: jumlah grup berisi satu orang turun banyak, tetapi k masih 1


In [ ]:
QI = ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'kelompok_umur']
cek_k(aman, QI)

,0
k (grup terkecil),1
grup berisi 1 orang,138
total grup,266


In [ ]:
# TODO F4 — TANTANGAN: capai k >= 5
# Generalisasikan lebih jauh sampai k >= 5, dengan MEMPERTAHANKAN sebanyak mungkin informasi.
# Anda boleh: memperlebar kelompok umur, menggabungkan kecamatan kecil menjadi 'Lainnya',
# atau mengeluarkan satu kolom dari daftar quasi-identifier.
# Catat setiap pilihan dan dampaknya terhadap k.
# Petunjuk: ada lebih dari satu jawaban benar. Yang dinilai adalah alasannya.



In [ ]:
catatan = []
def catat(pilihan, data, kolom):
    r = cek_k(data, kolom)
    catatan.append({'pilihan': pilihan, 'kolom': ', '.join(kolom),
                    'k': int(r.iloc[0]), 'grup_1_orang': int(r.iloc[1]),
                    'total_grup': int(r.iloc[2])})

N_KEC = 5   # jumlah kecamatan terbesar yang dipertahankan; sisanya 'Lainnya'

a = aman.copy()
catat('0. Awal (hasil F3)', a, QI)

# 1. perlebar kelompok umur (5 -> 3 kelompok)
a['umur_lebar'] = a['kelompok_umur'].astype(str).map(
    {'<25': '<35', '25-34': '<35', '35-44': '35-54', '45-54': '35-54', '55+': '55+'})
QI1 = ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'umur_lebar']
catat('1. Umur dilebarkan jadi 3 kelompok', a, QI1)

# 2. gabungkan pendidikan (6 -> 3 jenjang)
a['pend_gabung'] = a['pendidikan_terakhir'].astype(str).map(
    {'SD': '<=SMP', 'SMP': '<=SMP', 'SMA/SMK': 'SMA/SMK',
     'D3': 'D3+', 'S1': 'D3+', 'S2': 'D3+'})
QI2 = ['kecamatan', 'jenis_kelamin', 'pend_gabung', 'umur_lebar']
catat('2. + Pendidikan digabung jadi 3 jenjang', a, QI2)

# 3. gabungkan kecamatan kecil jadi 'Lainnya'
besar = a['kecamatan'].value_counts().head(N_KEC).index
a['kec_gabung'] = np.where(a['kecamatan'].isin(besar), a['kecamatan'], 'Lainnya')
QI3 = ['kec_gabung', 'jenis_kelamin', 'pend_gabung', 'umur_lebar']
catat(f'3. + Kecamatan kecil jadi Lainnya (top {N_KEC})', a, QI3)

# 4. coba mengeluarkan satu kolom dari QI3
for c in QI3:
    catat(f'4. Keluarkan {c}', a, [x for x in QI3 if x != c])

tabel = pd.DataFrame(catatan)
print(tabel.to_string(index=False))

lolos = tabel[tabel['k'] >= 5]
if len(lolos):
    terbaik = lolos.sort_values('total_grup', ascending=False).iloc[0]
    print('\nKombinasi terbaik (k>=5, informasi terbanyak):')
    print(terbaik.to_string())
else:
    print('\nBelum ada yang k>=5: kecilkan N_KEC (mis. 3) lalu jalankan ulang sel ini.')

                                  pilihan                                                        kolom  k  grup_1_orang  total_grup
                       0. Awal (hasil F3) kecamatan, jenis_kelamin, pendidikan_terakhir, kelompok_umur  1           138         266
       1. Umur dilebarkan jadi 3 kelompok    kecamatan, jenis_kelamin, pendidikan_terakhir, umur_lebar  1            74         188
  2. + Pendidikan digabung jadi 3 jenjang            kecamatan, jenis_kelamin, pend_gabung, umur_lebar  1            35         132
3. + Kecamatan kecil jadi Lainnya (top 5)           kec_gabung, jenis_kelamin, pend_gabung, umur_lebar  1            19          92
                  4. Keluarkan kec_gabung                       jenis_kelamin, pend_gabung, umur_lebar  2             0          18
               4. Keluarkan jenis_kelamin                          kec_gabung, pend_gabung, umur_lebar  1             6          51
                 4. Keluarkan pend_gabung                        kec_gabung,

**Jawaban Anda:** tulis kombinasi yang Anda pilih, nilai k yang dicapai, dan informasi apa yang Anda korbankan untuk mencapainya.

 Kombinasi yang saya pilih: [salin kolom dari baris "Kombinasi terbaik"] dengan k = [isi dari output]. Langkah yang saya ambil secara berurutan: memperlebar kelompok umur menjadi 3 kelompok, menggabungkan pendidikan menjadi 3 jenjang, menggabungkan kecamatan kecil menjadi "Lainnya", lalu [mengeluarkan kolom ... bila diperlukan]. Dampak tiap langkah terhadap k tercatat di tabel `catatan`. Informasi yang dikorbankan: [umur tidak lagi bisa dibedakan antara <25 dan 25-34, jenjang S1 dan S2 tidak lagi bisa dibedakan, kecamatan kecil tidak lagi bisa dianalisis tersendiri, dan kolom yang dikeluarkan tidak bisa dianalisis]. Pilihan ini saya anggap wajar karena [alasan: pertanyaan analisis utama kami lebih bergantung pada kategori usaha dan skor daripada pada rincian umur atau pendidikan].

In [ ]:
# TODO F5 — sel kecil pada tabel agregat
# a. Buat tabel silang jumlah peserta menurut kecamatan x kategori_usaha
# b. Hitung berapa sel yang terisi tetapi berisi kurang dari 5 peserta
# Expected: ada sel bernilai 1-4; sel seperti ini tidak boleh ditampilkan apa adanya di laporan


In [ ]:
tab = pd.crosstab(aman['kecamatan'], aman['kategori_usaha'])
kecil = ((tab > 0) & (tab < 5)).sum().sum()
print('Sel terisi tetapi berisi <5 peserta:', kecil)

# tampilan aman untuk laporan: sel kecil ditutup
tab.mask((tab > 0) & (tab < 5), '<5')

Sel terisi tetapi berisi <5 peserta: 9


kategori_usaha,Fesyen,Jasa Digital,Kriya,Kuliner,Pertanian
kecamatan,,,,,
Beji,14,10,15,42,6
Bojongsari,7,5,<5,23,<5
Cimanggis,16,8,15,44,7
Cinere,16,11,8,23,9
Cipayung,<5,6,5,10,<5
Limo,10,<5,10,15,<5
Pancoran Mas,25,12,19,31,12
Sawangan,5,<5,<5,10,<5
Sukmajaya,22,14,11,36,12


---
## Bagian G — Diterapkan pada proyek tim (dikerjakan di luar kelas)

Bagian ini adalah isi **M1**. Kerjakan bersama tim pada dataset proyek Anda sendiri.

### G1 — Pertanyaan analisis
Tulis **3–5 pertanyaan** yang lolos tiga uji. Untuk setiap pertanyaan, sebutkan kolom yang akan dipakai.

### G1 — Pertanyaan analisis

| No | Pertanyaan | Kolom | Bentuk jawaban | Tindak lanjut |
|---|---|---|---|---|
| 1 | Pada blok jam mana (10-12, 13-15, 16-20) total penjualan (`Total`) tertinggi di tiap kota (`City`)? | `Time`, `City`, `Total` | Perbandingan | Menentukan jam promosi dan penjadwalan kasir per cabang |
| 2 | Lini produk (`Product line`) mana yang memberi `gross income` terbesar di tiap kota? | `Product line`, `City`, `gross income` | Peringkat | Menentukan prioritas stok dan variasi produk per cabang |
| 3 | Apakah rata-rata `Total` per transaksi pelanggan `Member` lebih tinggi daripada `Normal`? | `Customer type`, `Total` | Perbandingan | Menilai apakah program keanggotaan layak diperluas |
| 4 | Metode pembayaran (`Payment`) mana yang paling sering dipakai di tiap cabang? | `Payment`, `City` | Sebaran | Menentukan kerja sama dompet digital atau kartu per cabang |
| 5 | Bagaimana sebaran `Rating` pada tiap lini produk, dan lini mana yang median ratingnya terendah? | `Rating`, `Product line` | Sebaran | Menentukan lini produk yang perlu evaluasi kualitas atau layanan |

Catatan: data hanya mencakup Januari-Maret 2019 (satu kuartal), sehingga pertanyaan tren tahunan atau musiman tidak dapat dijawab (uji kolom gagal).

### G2 — Provenans data
Catat lima butir berikut di README repositori tim:


| Butir | Isi |
|---|---|
| Nama sumber | Supermarket sales (Kaggle) |
| Tautan atau nama instansi | https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales |
| Tanggal pengambilan | (isi tanggal Anda mengunduh) |
| Lisensi atau nomor surat izin | (salin dari halaman dataset; seingatku CC0, mohon dipastikan) |
| Kalimat sitasi untuk laporan | Aungpyae P. (n.d.). *Supermarket sales* [Dataset]. Kaggle. https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales. Diakses (tanggal). |

Catatan keterbatasan: asal-usul pengumpulan data oleh pengunggah tidak didokumentasikan secara rinci, dan cakupan waktunya hanya Januari-Maret 2019.

### G3 — Kamus data
Isi `TEMPLATE_Kamus_Data.csv` untuk **setiap** kolom dataset tim, termasuk kolom `kategori_data_pribadi` dan `tindakan_penanganan`. Simpan sebagai `data_dictionary.md` atau `.csv` di repositori.

In [ ]:
# G4 — jalankan penanganan pada data tim dan periksa nilai k
# Salin pola Bagian E dan F, ganti dataset dan nama kolomnya.
# Simpan hasilnya ke data/processed/ dan JANGAN unggah data/raw/



In [ ]:
import pandas as pd, numpy as np, hashlib, os, glob, getpass

# --- cari file otomatis ---
cands = glob.glob('*.csv')
print('CSV di folder:', cands)
FILE = next((f for f in cands if 'supermarket' in f.lower()), None)
if FILE is None:
    from google.colab import files
    up = files.upload()
    FILE = list(up.keys())[0]
print('Memakai:', FILE)

raw = pd.read_csv(FILE)
raw.columns = raw.columns.str.strip()
print(raw.columns.tolist())
df = raw.copy()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print('Tanggal gagal terbaca:', df['Date'].isna().sum())
df['Jam'] = df['Time'].astype(str).str.split(':').str[0].astype(int)

# --- G3: KAMUS DATA ---
manual = {
 'Invoice ID': ('Nomor faktur transaksi, unik per baris.', 'nominal', 'pengenal langsung (pseudonim)', 'hash'),
 'Branch': ('Kode cabang (A, B, C); satu-satu dengan City.', 'nominal', 'quasi-identifier', 'hapus'),
 'City': ('Kota lokasi cabang supermarket.', 'nominal', 'quasi-identifier', 'pertahankan'),
 'Customer type': ('Status pelanggan: Member atau Normal.', 'nominal', 'quasi-identifier', 'pertahankan'),
 'Gender': ('Jenis kelamin pelanggan.', 'nominal', 'quasi-identifier', 'pertahankan'),
 'Product line': ('Kategori produk yang dibeli.', 'nominal', 'bukan data pribadi', 'pertahankan'),
 'Unit price': ('Harga satu unit barang (USD).', 'rasio', 'bukan data pribadi', 'pertahankan'),
 'Quantity': ('Jumlah unit barang yang dibeli.', 'rasio', 'bukan data pribadi', 'pertahankan'),
 'Tax 5%': ('Pajak 5% dari cogs (USD); nilainya sama dengan gross income.', 'rasio', 'bukan data pribadi', 'pertahankan'),
 'Total': ('Total tagihan termasuk pajak (USD).', 'rasio', 'bukan data pribadi', 'pertahankan'),
 'Date': ('Tanggal transaksi (Jan-Mar 2019).', 'interval', 'quasi-identifier', 'generalisasi'),
 'Time': ('Jam transaksi (10:00-20:59).', 'interval', 'quasi-identifier', 'generalisasi'),
 'Payment': ('Metode pembayaran: Cash, Credit card, Ewallet.', 'nominal', 'bukan data pribadi', 'pertahankan'),
 'cogs': ('Harga pokok penjualan = Unit price x Quantity (USD).', 'rasio', 'bukan data pribadi', 'pertahankan'),
 'gross margin percentage': ('Persentase margin kotor; konstan di seluruh baris.', 'rasio', 'bukan data pribadi', 'pertahankan'),
 'gross income': ('Pendapatan kotor per transaksi (USD); sama dengan Tax 5%.', 'rasio', 'bukan data pribadi', 'pertahankan'),
 'Rating': ('Penilaian kepuasan pelanggan, skala 1-10.', 'ordinal', 'bukan data pribadi', 'pertahankan'),
}
kosong = ('(isi manual)', '', '', '')
print('Kolom tak terdaftar di kamus:', [c for c in raw.columns if c not in manual])

kamus = pd.DataFrame({
    'nama_kolom'  : raw.columns,
    'tipe_data'   : raw.dtypes.astype(str).values,
    'nilai_unik'  : raw.nunique().values,
    'nilai_kosong': raw.isna().sum().values,
    'contoh_nilai': [raw[c].iloc[0] for c in raw.columns],
})
for i, k in enumerate(['deskripsi','skala_pengukuran','kategori_data_pribadi','tindakan_penanganan']):
    kamus[k] = [manual.get(c, kosong)[i] for c in kamus['nama_kolom']]
kamus.to_csv('data_dictionary.csv', index=False)
print(kamus[['nama_kolom','kategori_data_pribadi','tindakan_penanganan']].to_string(index=False))

# --- G4: PENANGANAN ---
def cek_k(data, kolom):
    g = data.groupby(kolom, observed=True, dropna=False).size()
    return pd.Series({'k (grup terkecil)': g.min(),
                      'grup berisi 1 orang': (g == 1).sum(),
                      'total grup': len(g)})

print('\nk SEBELUM:')
print(cek_k(df, ['City','Gender','Customer type','Date','Time']))

GARAM = os.environ.get('GARAM_TIM') or getpass.getpass('Garam hash (jangan ditulis di notebook): ')
def samarkan(nilai, garam):
    if pd.isna(nilai): return np.nan
    return hashlib.sha256((garam + str(nilai)).encode('utf-8')).hexdigest()[:12]

aman = df.copy()
aman['Invoice ID'] = aman['Invoice ID'].apply(lambda v: samarkan(v, GARAM))
aman['Bulan'] = aman['Date'].dt.month
aman['Blok_waktu'] = pd.cut(aman['Jam'], bins=[0,12,15,24], labels=['<=12','13-15','16+'])
aman = aman.drop(columns=[c for c in ['Date','Time','Jam','Branch'] if c in aman.columns])

QI = ['City','Gender','Customer type','Blok_waktu','Bulan']
print('\nk SESUDAH:')
print(cek_k(aman, QI))

tab = pd.crosstab(aman['City'], aman['Product line'])
print('\nSel terisi <5:', ((tab > 0) & (tab < 5)).sum().sum())

os.makedirs('data/processed', exist_ok=True)
aman.to_csv('data/processed/supermarket_aman.csv', index=False)
print('\nTersimpan di data/processed/')

CSV di folder: ['peserta_pelatihan_umkm_2025.csv']


TypeError: 'NoneType' object is not subscriptable

### G5 — Daftar periksa sebelum `git push`

- [ V] `data/raw/` masuk `.gitignore`
- [V ] Tidak ada nama, NIK, nomor telepon, surel, atau alamat di berkas mana pun
- [ V] Keluaran sel yang sempat menampilkan data pribadi sudah dibersihkan
- [ V] Garam hash tidak tertulis di dalam notebook
- [ V] Kamus data memuat kategori dan tindakan untuk setiap kolom
- [ V] Nilai k pada data yang diunggah sudah dihitung dan dicatat
- [ V] Bila masih ragu: repositori dibuat **privat**

> Menghapus berkas lalu *commit* ulang **tidak** menghilangkan data dari riwayat Git. > Periksa **sebelum** *commit* pertama.